In [ ]:
import arcpy, os, json, traceback
from pathlib import Path
from datetime import datetime
from collections import defaultdict

arcpy.env.overwriteOutput = True
print(f"\u2705 ArcPy version : {arcpy.GetInstallInfo()['Version']}")
print(f"\u2705 Script started : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# ─── USER CONFIGURATION ─────────────────────────────────────────────────────────
ROOT_FOLDER  = r"C:\path\to\input_folder"
SCRATCH_GDB  = r"C:\path\to\scratch.gdb"
SUPPORTED_EXTENSIONS = {".kml", ".kmz", ".shp"}
SKIP_EXTENSIONS      = {".pdf", ".lyr", ".lyrx", ".mxd", ".aprx"}
# ─────────────────────────────────────────────────────────────────────────
print(f"\ud83d\udcc1 Root folder : {ROOT_FOLDER}")
print(f"\ud83d\udce6 GDB         : {SCRATCH_GDB}")

In [ ]:
def ensure_gdb(gdb_path):
    """Ensure the file geodatabase exists; create if needed."""
    if gdb_path and arcpy.Exists(gdb_path):
        print(f"   \ud83d\udcc2 Using existing GDB: {gdb_path}")
        return gdb_path
    if gdb_path is None:
        gdb_path = os.path.join(os.getcwd(), "scratch_imports.gdb")
    parent, name = str(Path(gdb_path).parent), Path(gdb_path).name
    arcpy.management.CreateFileGDB(parent, name)
    print(f"   \ud83d\udcc2 Created GDB: {gdb_path}")
    return gdb_path


def find_spatial_files(root, supported, skip):
    """Walk root and return (kml, kmz, shp, skipped_count, unknown_exts)."""
    kml, kmz, shp = [], [], []
    skipped = 0
    unknown = set()
    seen = set()
    shapefile_sidecars = {".dbf", ".prj", ".sbn", ".sbx", ".shx", ".cpg", ".xml"}
    for dp, dns, fns in os.walk(root):
        dns[:] = [d for d in dns if not d.startswith('.')]
        for fn in fns:
            ext = Path(fn).suffix.lower()
            fp = os.path.join(dp, fn)
            if ext in skip:
                skipped += 1
            elif ext == ".kml":  kml.append(fp)
            elif ext == ".kmz":  kmz.append(fp)
            elif ext == ".shp":
                key = os.path.join(dp, Path(fn).stem)
                if key not in seen:
                    shp.append(fp); seen.add(key)
            elif ext not in supported and ext not in shapefile_sidecars:
                unknown.add(ext)
    return kml, kmz, shp, skipped, unknown


def get_subfolder(filepath, root):
    """Return the immediate subfolder name under root for grouping."""
    parts = Path(os.path.relpath(filepath, root)).parts
    return parts[0] if len(parts) > 1 else "_Root_Files"


def create_group_lyrx(name, out_dir):
    """Write a minimal group-layer .lyrx template and return its path."""
    safe = name.replace(' ', '_').replace('-', '_')
    doc = {
        "type": "CIMLayerDocument", "version": "3.0.0",
        "layers": [f"CIMPATH=map/{safe}.json"],
        "layerDefinitions": [{
            "type": "CIMGroupLayer",
            "name": name,
            "uRI": f"CIMPATH=map/{safe}.json",
            "visibility": True, "layers": [], "expanded": True
        }]
    }
    p = os.path.join(out_dir, f"{safe}.lyrx")
    with open(p, 'w') as f:
        json.dump(doc, f, indent=2)
    return p


def get_or_create_group(active_map, name, temp_dir):
    """Return an existing group layer or create a new one."""
    for l in active_map.listLayers():
        if l.isGroupLayer and l.name == name:
            return l
    lyrx = create_group_lyrx(name, temp_dir)
    lf = arcpy.mp.LayerFile(lyrx)
    active_map.addLayer(lf, "TOP")
    for l in active_map.listLayers():
        if l.isGroupLayer and l.name == name:
            return l
    raise RuntimeError(f"Could not create group layer: {name}")


def add_data_to_group(active_map, data_path, group_layer):
    """Add data to map, then move the new layer(s) into a group."""
    before = {l.longName for l in active_map.listLayers()}
    active_map.addDataFromPath(data_path)
    new_layers = [l for l in active_map.listLayers()
                  if l.longName not in before and 'C:\path\to\folder' not in l.longName]
    for lyr in new_layers:
        active_map.addLayerToGroup(group_layer, lyr)
        active_map.removeLayer(lyr)
    return len(new_layers)


print("\u2705 Helper functions defined.")

In [ ]:
if not os.path.isdir(ROOT_FOLDER):
    raise FileNotFoundError(f"Root folder not found: {ROOT_FOLDER}")

kml_files, kmz_files, shp_files, skipped_count, unknown_exts = find_spatial_files(
    ROOT_FOLDER, SUPPORTED_EXTENSIONS, SKIP_EXTENSIONS)

total = len(kml_files) + len(kmz_files) + len(shp_files)
print(f"{'\u2500'*60}")
print(f"  KML files      : {len(kml_files):>4}")
print(f"  KMZ files      : {len(kmz_files):>4}")
print(f"  Shapefiles     : {len(shp_files):>4}")
print(f"  Total to import: {total:>4}")
print(f"  Skipped (PDF+) : {skipped_count:>4}")
if unknown_exts:
    print(f"  Unknown exts   : {', '.join(sorted(unknown_exts))}")
print(f"{'\u2500'*60}")

# Group files by subfolder
grouped = defaultdict(list)
for f in kml_files + kmz_files + shp_files:
    grouped[get_subfolder(f, ROOT_FOLDER)].append(f)

print(f"\n\ud83d\udcc2 {len(grouped)} project folder(s) detected:")
for folder, files in sorted(grouped.items()):
    print(f"   \u251c\u2500 {folder} ({len(files)} files)")

In [ ]:
SCRATCH_GDB = ensure_gdb(SCRATCH_GDB)
TEMP_DIR = os.path.join(os.path.dirname(SCRATCH_GDB), "_temp_lyrx")
os.makedirs(TEMP_DIR, exist_ok=True)
print(f"\u2705 GDB ready: {SCRATCH_GDB}")
print(f"\u2705 Temp dir : {TEMP_DIR}")

In [ ]:
try:
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    active_map = aprx.activeMap
    if active_map is None:
        all_maps = aprx.listMaps()
        if not all_maps:
            raise RuntimeError("No maps in project.")
        active_map = all_maps[0]
        print(f"\u26a0\ufe0f  Using first map: '{active_map.name}'")
    else:
        print(f"\u2705 Active map : '{active_map.name}'")
    print(f"\u2705 Project    : {aprx.filePath}")
except Exception as e:
    print(f"\u274c {e}"); raise

In [ ]:
results = {"success": [], "failed": []}

for folder_name in sorted(grouped.keys()):
    files = grouped[folder_name]
    print(f"\n{'='*60}")
    print(f"  \ud83d\udcc2 {folder_name}  ({len(files)} file(s))")
    print(f"{'='*60}")

    # Create or find the group layer for this subfolder
    try:
        group_lyr = get_or_create_group(active_map, folder_name, TEMP_DIR)
        print(f"   \ud83d\udcc1 Group layer ready: {folder_name}")
    except Exception as e:
        print(f"   \u274c Could not create group '{folder_name}': {e}")
        for f in files:
            results["failed"].append((f, str(e)))
        continue

    # Add each file into the group
    for filepath in files:
        rel = os.path.relpath(filepath, ROOT_FOLDER)
        try:
            add_data_to_group(active_map, filepath, group_lyr)
            print(f"   \u2705 {rel}")
            results["success"].append(filepath)
        except Exception as e:
            print(f"   \u274c {rel}\n       \u21b3 {e}")
            results["failed"].append((filepath, str(e)))

print(f"\n{'='*60}")
print("  Import complete!")
print(f"{'='*60}")

In [ ]:
print(f"\n{'\u2550'*60}")
print("  \ud83d\udcca  IMPORT SUMMARY")
print(f"{'\u2550'*60}")
print(f"  \u2705 Added  : {len(results['success'])}")
print(f"  \u274c Failed : {len(results['failed'])}")
if results["failed"]:
    print(f"\n  \u2500\u2500 Failed files:")
    for p, err in results["failed"]:
        print(f"  \u2022 {os.path.relpath(p, ROOT_FOLDER)}")
        print(f"    \u21b3 {err}")
print(f"\n  GDB      : {SCRATCH_GDB}")
print(f"  Finished : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'\u2550'*60}")

# Clean up temp .lyrx files
import shutil
if os.path.isdir(TEMP_DIR):
    shutil.rmtree(TEMP_DIR, ignore_errors=True)
    print("\ud83e\uddf9 Cleaned up temp .lyrx files.")

try:
    aprx.save()
    print("\ud83d\udcbe ArcGIS Pro project saved.")
except Exception as e:
    print(f"\u26a0\ufe0f  Could not save: {e}")

In [ ]:
# ── 1. Folders processed ──────────────────────────────────────────────────
all_subfolders = sorted(
    d for d in os.listdir(ROOT_FOLDER)
    if os.path.isdir(os.path.join(ROOT_FOLDER, d)) and not d.startswith('.')
)

print(f"{'='*60}")
print(f"  \ud83d\udcc2  FOLDERS PROCESSED  ({len(all_subfolders)})")
print(f"{'='*60}")
for i, folder in enumerate(all_subfolders, 1):
    print(f"  {i:>3}. {folder}")

# ── 2. PDF inventory ─────────────────────────────────────────────────────
pdf_by_project = defaultdict(list)
for dirpath, dirnames, filenames in os.walk(ROOT_FOLDER):
    dirnames[:] = [d for d in dirnames if not d.startswith('.')]
    for fn in filenames:
        if Path(fn).suffix.lower() == '.pdf':
            project = get_subfolder(os.path.join(dirpath, fn), ROOT_FOLDER)
            pdf_by_project[project].append(fn)

total_pdfs = sum(len(v) for v in pdf_by_project.values())

print(f"\n{'='*60}")
print(f"  \ud83d\udcc4  PDF FILES FOUND  ({total_pdfs} total)")
print(f"{'='*60}")

if total_pdfs == 0:
    print("  (none)")
else:
    print(f"  {'Project':<35} {'PDF File Name'}")
    print(f"  {'\u2500'*35} {'\u2500'*40}")
    for project in sorted(pdf_by_project.keys()):
        for pdf_name in sorted(pdf_by_project[project]):
            print(f"  {project:<35} {pdf_name}")

print(f"{'='*60}")